In [23]:
import torch
torch.cuda.empty_cache()

In [24]:
torch.cuda.ipc_collect()

In [25]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [26]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [27]:
!kill -9 626

/bin/bash: line 1: kill: (626) - No such process


In [28]:
torch.cuda.empty_cache()

In [29]:
import torch
import torch.nn as nn
import torchvision.models as models
from timm import create_model

class ResViT(nn.Module):
    def __init__(self, num_classes=4):
        super(ResViT, self).__init__()

        # 🔹 ResNet18 (feature extractor)
        self.cnn = models.resnet18(weights="IMAGENET1K_V1")
        cnn_dim = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity()

        # 🔹 ViT Small (feature extractor)
        self.vit = create_model('vit_small_patch16_224', pretrained=True, num_classes=0)
        vit_dim = 384

        # 🔹 Fusion dimension
        self.feature_dim = cnn_dim + vit_dim

        # 🔥 (OPTIONAL BUT BETTER) Gated Fusion
        self.gate = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim),
            nn.Sigmoid()
        )

        # 🔹 Classifier
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(self.feature_dim),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # CNN features
        f_cnn = self.cnn(x)          # shape: [B, cnn_dim]

        # ViT features
        f_vit = self.vit(x)          # shape: [B, vit_dim]

        # 🔹 Concatenation
        f = torch.cat([f_cnn, f_vit], dim=1)

        # 🔥 Gated Fusion (improves performance)
        g = self.gate(f)
        f = f * g

        # Classification
        out = self.classifier(f)

        return out

In [30]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ResViT().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [31]:
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
import torch
import gc

scaler = GradScaler()

def train_model(model, train_loader, val_loader, epochs=20):
    best_acc = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in tqdm(train_loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            # ✅ MIXED PRECISION START
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            # ✅ MIXED PRECISION END

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            # ✅ OPTIONAL: free small tensors
            del outputs, loss

        scheduler.step()

        # ✅ Evaluation
        val_acc = evaluate(model, val_loader)

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Val Acc: {val_acc:.4f}")

        # ✅ MEMORY CLEANUP AFTER EACH EPOCH
        gc.collect()
        torch.cuda.empty_cache()

    return model

/tmp/ipykernel_55/2102352208.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [32]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return correct / total

In [33]:
!pip install grad-cam

In [34]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Use last conv layer of ResNet
target_layers = [model.cnn.layer4[-1]]

cam = GradCAM(model=model, target_layers=target_layers)

def apply_gradcam(image_tensor):
    grayscale_cam = cam(input_tensor=image_tensor.unsqueeze(0))

    cam_image = show_cam_on_image(
        image_tensor.permute(1,2,0).cpu().numpy(),
        grayscale_cam[0],
        use_rgb=True
    )

    return cam_image

In [35]:
data_dir = "/kaggle/input/datasets/venkatsaikondra/multi-venkat/Final_Data"

In [36]:
from torchvision import datasets, transforms

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = datasets.ImageFolder(
    data_dir + "/train", transform=train_transform
)

val_dataset = datasets.ImageFolder(
    data_dir + "/val", transform=val_transform
)

test_dataset = datasets.ImageFolder(
    data_dir + "/test", transform=val_transform
)

In [37]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [38]:
model = train_model(model, train_loader, val_loader, epochs=20)

test_acc = evaluate(model, test_loader)

print(f"Final Test Accuracy: {test_acc:.4f}")

  0%|          | 0/472 [00:00<?, ?it/s]/tmp/ipykernel_55/2102352208.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 472/472 [01:09<00:00,  6.76it/s]



Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       404
           1       0.90      0.96      0.93       404
           2       0.83      0.73      0.77       404
           3       0.76      0.79      0.77       404

    accuracy                           0.87      1616
   macro avg       0.87      0.87      0.87      1616
weighted avg       0.87      0.87      0.87      1616


Confusion Matrix:
[[403   0   0   1]
 [  3 386   1  14]
 [  1  21 294  88]
 [  2  22  60 320]]
Epoch 1, Loss: 359.5184, Val Acc: 0.8682


  0%|          | 0/472 [00:00<?, ?it/s]/tmp/ipykernel_55/2102352208.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 472/472 [01:09<00:00,  6.79it/s]



Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       404
           1       0.91      0.98      0.94       404
           2       0.82      0.80      0.81       404
           3       0.80      0.76      0.78       404

    accuracy                           0.88      1616
   macro avg       0.88      0.88      0.88      1616
weighted avg       0.88      0.88      0.88      1616


Confusion Matrix:
[[401   0   0   3]
 [  0 397   4   3]
 [  0  11 323  70]
 [  1  30  65 308]]
Epoch 2, Loss: 294.7090, Val Acc: 0.8843


  0%|          | 0/472 [00:00<?, ?it/s]/tmp/ipykernel_55/2102352208.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 472/472 [01:09<00:00,  6.79it/s]


...


Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       404
           1       0.98      0.99      0.98       404
           2       0.85      0.85      0.85       404
           3       0.85      0.84      0.84       404

    accuracy                           0.92      1616
   macro avg       0.92      0.92      0.92      1616
weighted avg       0.92      0.92      0.92      1616


Confusion Matrix:
[[403   0   0   1]
 [  0 399   3   2]
 [  0   3 345  56]
 [  1   6  59 338]]
Epoch 18, Loss: 199.2891, Val Acc: 0.9189


  0%|          | 0/472 [00:00<?, ?it/s]/tmp/ipykernel_55/2102352208.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 472/472 [01:09<00:00,  6.84it/s]



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       404
           1       0.98      0.98      0.98       404
           2       0.88      0.82      0.85       404
           3       0.83      0.88      0.85       404

    accuracy                           0.92      1616
   macro avg       0.92      0.92      0.92      1616
weighted avg       0.92      0.92      0.92      1616


Confusion Matrix:
[[403   0   0   1]
 [  1 397   2   4]
 [  0   3 331  70]
 [  1   5  44 354]]
Epoch 19, Loss: 197.5425, Val Acc: 0.9189


  0%|          | 0/472 [00:00<?, ?it/s]/tmp/ipykernel_55/2102352208.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 472/472 [01:09<00:00,  6.76it/s]



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       404
           1       0.98      0.99      0.98       404
           2       0.84      0.89      0.86       404
           3       0.88      0.82      0.85       404

    accuracy                           0.92      1616
   macro avg       0.92      0.92      0.92      1616
weighted avg       0.92      0.92      0.92      1616


Confusion Matrix:
[[403   0   0   1]
 [  0 398   3   3]
 [  0   2 360  42]
 [  1   5  67 331]]
Epoch 20, Loss: 197.2168, Val Acc: 0.9233

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       405
           1       0.99      0.98      0.99       405
           2       0.85      0.87      0.86       405
           3       0.86      0.85      0.85       405

    accuracy                           0.92      1620
   macro avg       0.92      0.92      0.92      1620